# RouterQueryEngine — Picking the Right Index at Query Time

Sometimes you have multiple indices built different ways for different purposes — one tuned for precise fact lookup, another tuned for broad summaries. A `RouterQueryEngine` lets the LLM itself pick which index to query based on the question, instead of you hardcoding that decision.


Standard setup: quiet logging, load env vars, and set the default LLM/embedding model.


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet noisy INFO-level logs from LlamaIndex and its HTTP client.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Load API keys from .env into the environment.
load_dotenv()

# Set the default LLM and embedding model used everywhere in this notebook.
Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 1 — Build two different indices from the same documents.** A `VectorStoreIndex` for precise fact lookup and a `SummaryIndex` for broad overviews — the router will choose between them per question later.


In [2]:
from llama_index.core import SimpleDirectoryReader, SummaryIndex, VectorStoreIndex

# Load the same anime corpus once, then build two different index types from it.
documents = SimpleDirectoryReader("data/sample_docs").load_data()

# Good for precise fact lookup — embeds nodes and retrieves the most similar ones.
vector_index = VectorStoreIndex.from_documents(documents)

# Good for broad summaries — reads through every node rather than retrieving a subset.
summary_index = SummaryIndex.from_documents(documents)

**Step 2 — Wrap each index as a tool with a description.** `QueryEngineTool` pairs a query engine with a natural-language description; the router's selector LLM reads _only_ these descriptions to decide which tool to use, so wording them well matters.


In [3]:
from llama_index.core.tools import QueryEngineTool

# Wrap each index as a QueryEngineTool with a clear description — the router's
# selector LLM reads ONLY these descriptions to decide which tool fits a question.
vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_index.as_query_engine(),
    description=(
        "Useful for answering a specific, narrow factual question about one particular anime — for example a character's signature technique, a studio, or a specific rule from the story."
    ),
)

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_index.as_query_engine(response_mode="tree_summarize"),
    description=(
        "Useful for broad questions that ask to summarize, compare, or give an overview across ALL of these anime series at once, rather than one single specific fact."
    ),
)

**Step 3 — Let the LLM route between tools.** `RouterQueryEngine` picks one tool per question using an `LLMSingleSelector`, then queries whichever one it picked — run two very different questions to watch it choose differently each time.


In [5]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.llms.openai import OpenAI

# The selector LLM only makes one structured decision (which tool to pick), but
# that decision needs to be reliable — gpt-4o-mini is used here instead of the
# project-wide gpt-4.1-nano default for a more dependable routing choice.
router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(llm=OpenAI(model="gpt-4o-mini")),
    query_engine_tools=[vector_tool, summary_tool],
)

tool_names = ["vector_tool (specific lookup)", "summary_tool (broad overview)"]
questions = [
    "What is Naruto's signature technique?",
    "Give me a high-level overview of all five anime series covered here — their protagonists, studios, and themes.",
]

for question in questions:
    response = router_engine.query(question)  # the router picks a tool internally, then queries it
    # selector_result records which tool the LLM chose and why — handy for debugging routing.
    selection = response.metadata["selector_result"].selections[0]
    print(f"Q: {question}")
    print(f"Routed to: {tool_names[selection.index]}")
    print(f"Reason: {selection.reason}")
    print(f"A: {response}\n")

Q: What is Naruto's signature technique?
Routed to: vector_tool (specific lookup)
Reason: The question asks for a specific fact about Naruto, which is his signature technique.
A: Naruto's signature technique is the Rasengan, a swirling ball of concentrated chakra.

Q: Give me a high-level overview of all five anime series covered here — their protagonists, studios, and themes.
Routed to: summary_tool (broad overview)
Reason: The question asks for a high-level overview of all five anime series, which aligns with the need for broad comparisons and summaries across multiple series.
A: The five anime series are "Death Note," "Demon Slayer," "Dragon Ball," "Naruto," and "Solo Leveling." 

"Death Note" follows Light Yagami, a highly intelligent high school student who discovers a supernatural notebook that allows him to kill anyone by writing their name. The series is animated by Studio Madhouse and explores themes of vigilante justice, power corruption, and psychological warfare.

"Demon Sl

### Summary

- A `RouterQueryEngine` turns "which index do I use?" into a decision the system makes automatically, based on each `QueryEngineTool`'s description, instead of you hardcoding it.
- The tool descriptions matter as much as the indices themselves — they're the only thing the selector LLM reads to decide.
